# 🏗️ 第四课：多领域实战案例 — OPERA 框架举一反三

> **用同一个 OPERA 框架，对四个完全不同的行业场景进行建模，证明它真的是"万能"框架。**

### 📋 学习目标

| # | 目标 | 状态 |
|---|------|------|
| 1 | 独立完成智慧医院 OPERA 建模 + 代码 | ☐ |
| 2 | 独立完成电商供应链 OPERA 建模 + 代码 | ☐ |
| 3 | 独立完成智慧交通 OPERA 建模 + 代码 | ☐ |
| 4 | 独立完成企业知识管理 OPERA 建模 + 代码 | ☐ |
| 5 | 总结 OPERA 在不同领域的共性和差异 | ☐ |

### 前置要求
- 完成第一～三课（概念 + 框架 + 代码基础）

In [ ]:
# 🔧 通用工具：OntologyBuilder — 统一的建模辅助类
# 四个案例都复用这个工具，减少重复代码

class OntologyBuilder:
    """OPERA 建模辅助工具 — 支持快速建模并打印结构"""
    
    def __init__(self, name):
        self.name = name
        self.O = {}  # {class_name: {parent, desc, instances: []}}
        self.P = {}  # {class_name: [(prop, type, required, desc)]}
        self.E = []  # [{name, subject, object, precondition, effect}]
        self.R = []  # [(name, domain, range, cardinality, desc)]
        self.A = []  # [{type, desc, rule_text}]
        self._instances = {}  # {name: {class, props}}
    
    def add_class(self, name, parent=None, desc=""):
        self.O[name] = {"parent": parent, "desc": desc, "instances": []}
    
    def add_property(self, cls, name, dtype, required=True, desc=""):
        self.P.setdefault(cls, []).append((name, dtype, required, desc))
    
    def add_event(self, name, trigger, auto_action):
        self.E.append({"type": "event", "name": name, "trigger": trigger, "auto": auto_action})
    
    def add_action(self, name, subject, obj, precondition, effect):
        self.E.append({"type": "action", "name": name, "subject": subject, 
                       "object": obj, "precondition": precondition, "effect": effect})
    
    def add_relation(self, name, domain, range_cls, card="多对一", desc=""):
        self.R.append((name, domain, range_cls, card, desc))
    
    def add_axiom(self, atype, desc, rule_text):
        self.A.append({"type": atype, "desc": desc, "rule": rule_text})
    
    def add_instance(self, name, cls, **props):
        self._instances[name] = {"class": cls, "props": props}
        self.O.setdefault(cls, {}).setdefault("instances", []).append(name)
    
    def print_opera(self):
        """打印完整的 OPERA 模型"""
        print(f"\n{'='*65}")
        print(f"🎭 OPERA 模型：{self.name}")
        print(f"{'='*65}")
        
        # O
        print(f"\n📦 O — 对象类型 ({len(self.O)} 个类)")
        for name, info in self.O.items():
            parent = f" → {info.get('parent', '')}" if info.get('parent') else ""
            print(f"   · {name}{parent}  {info.get('desc', '')}")
        
        # P
        print(f"\n📋 P — 属性")
        for cls, props in self.P.items():
            print(f"   [{cls}]")
            for pname, dtype, req, desc in props:
                r = "✅" if req else "  "
                print(f"     {r} {pname}({dtype}) — {desc}")
        
        # E
        print(f"\n⚡ E — 事件与动作 ({len(self.E)} 个)")
        for e in self.E:
            if e["type"] == "action":
                print(f"   🎬 {e['name']}: {e['subject']} → {e['object']}")
                print(f"      前置: {e['precondition']}")
                print(f"      效果: {e['effect']}")
            else:
                print(f"   🔔 {e['name']}: 当 {e['trigger']} → {e['auto']}")
        
        # R
        print(f"\n🔗 R — 关系 ({len(self.R)} 条)")
        for name, domain, range_cls, card, desc in self.R:
            print(f"   · {domain} --[{name}]--> {range_cls}  ({card}) {desc}")
        
        # A
        print(f"\n⚖️ A — 公理 ({len(self.A)} 条)")
        icons = {"约束": "🔒", "推理": "💡", "互斥": "🚫"}
        for a in self.A:
            icon = icons.get(a["type"], "📌")
            print(f"   {icon} [{a['type']}] {a['desc']}")
            print(f"      {a['rule']}")
        
        total = len(self.O) + sum(len(v) for v in self.P.values()) + len(self.E) + len(self.R) + len(self.A)
        print(f"\n📊 模型规模：{len(self.O)} 类 / {sum(len(v) for v in self.P.values())} 属性 / {len(self.E)} 事件动作 / {len(self.R)} 关系 / {len(self.A)} 公理")
    
    def reason(self, rules):
        """运行推理规则"""
        print(f"\n🤖 推理引擎：{self.name}")
        print("-" * 50)
        findings = []
        for name, data in self._instances.items():
            for rule in rules:
                result = rule(name, data)
                if result:
                    findings.append(result)
                    print(f"  💡 {result}")
        if not findings:
            print("  (无推理结果)")
        print(f"  共发现 {len(findings)} 条推理结果")
        return findings

print("✅ OntologyBuilder 工具类已就绪")

---
## 🏥 案例 1：智慧医院

> **场景：** 一家三甲医院需要统一管理患者、医生、科室、药品、诊断和处方数据，
> 并自动检测用药安全风险。
>
> **核心问题：** 如何在开处方时自动检测药物过敏？如何自动发现高危患者？

In [ ]:
# 🏥 案例 1：智慧医院 OPERA 建模

hospital = OntologyBuilder("智慧医院")

# === O: 对象类型 ===
hospital.add_class("Person", desc="人")
hospital.add_class("Patient", parent="Person", desc="患者")
hospital.add_class("InPatient", parent="Patient", desc="住院患者")
hospital.add_class("OutPatient", parent="Patient", desc="门诊患者")
hospital.add_class("Doctor", parent="Person", desc="医生")
hospital.add_class("Nurse", parent="Person", desc="护士")
hospital.add_class("Department", desc="科室")
hospital.add_class("Ward", desc="病房")
hospital.add_class("Medication", desc="药物")
hospital.add_class("Prescription", desc="处方")
hospital.add_class("Diagnosis", desc="诊断")

# === P: 属性 ===
hospital.add_property("Patient", "patient_id", "string", True, "病历号")
hospital.add_property("Patient", "name", "string", True, "姓名")
hospital.add_property("Patient", "age", "int", True, "年龄")
hospital.add_property("Patient", "blood_type", "enum[A,B,AB,O]", True, "血型")
hospital.add_property("Patient", "allergies", "list[string]", False, "过敏药物列表")
hospital.add_property("Doctor", "doctor_id", "string", True, "工号")
hospital.add_property("Doctor", "specialty", "string", True, "专科")
hospital.add_property("Medication", "generic_name", "string", True, "通用名")
hospital.add_property("Medication", "contraindications", "list[string]", False, "禁忌症")
hospital.add_property("Prescription", "dosage", "string", True, "用法用量")

# === E: 事件与动作 ===
hospital.add_action("Prescribe", "Doctor", "Prescription", 
    "医生有处方权 + 患者无过敏冲突", "创建处方 + 过敏安全检查")
hospital.add_action("Admit", "Doctor", "Patient→InPatient", 
    "科室有空余床位", "患者入院 + 分配病房")
hospital.add_action("Discharge", "Doctor", "InPatient→OutPatient", 
    "治疗完成 + 费用结清", "出院 + 生成出院小结")
hospital.add_event("AllergyAlert", "处方药物在患者过敏列表中", "阻止处方 + 通知医生")
hospital.add_event("CriticalVitals", "患者生命体征超出安全范围", "触发紧急alert + 通知值班医生")

# === R: 关系 ===
hospital.add_relation("belongs_to", "Doctor", "Department", "多对一", "医生所属科室")
hospital.add_relation("admitted_to", "InPatient", "Ward", "多对一", "住院患者所在病房")
hospital.add_relation("diagnosed_with", "Patient", "Diagnosis", "一对多", "患者的诊断")
hospital.add_relation("prescribed_by", "Prescription", "Doctor", "多对一", "处方开具医生")
hospital.add_relation("prescribed_for", "Prescription", "Patient", "多对一", "处方针对患者")
hospital.add_relation("contains_med", "Prescription", "Medication", "多对多", "处方包含药物")
hospital.add_relation("ward_in", "Ward", "Department", "多对一", "病房所属科室")

# === A: 公理 ===
hospital.add_axiom("约束", "过敏安全约束", "∀ 处方中的药物 ∉ 患者过敏列表")
hospital.add_axiom("约束", "床位容量约束", "Ward.current_patients ≤ Ward.max_capacity")
hospital.add_axiom("推理", "高危患者识别", "age > 70 ∧ has_surgery → HighRiskPatient")
hospital.add_axiom("推理", "药物相互作用", "med_A.interacts_with(med_B) ∧ 同一处方 → 用药冲突警告")
hospital.add_axiom("互斥", "患者类型互斥", "InPatient ∩ OutPatient = ∅")

hospital.print_opera()

In [ ]:
# 🏥 智慧医院 — 用药安全推理实战

hospital.add_instance("张三", "Patient", age=75, allergies=["青霉素", "磺胺"], has_surgery=True)
hospital.add_instance("李四", "Patient", age=30, allergies=[], has_surgery=False)
hospital.add_instance("王五", "Patient", age=82, allergies=["阿司匹林"], has_surgery=True)
hospital.add_instance("赵六", "Patient", age=5, allergies=["头孢"], has_surgery=False)

rules = [
    # 高危患者识别
    lambda name, data: f"🚨 {name}: 高危患者（{data['props']['age']}岁 + 需手术）" 
        if data["class"] == "Patient" and data["props"].get("age", 0) > 70 
        and data["props"].get("has_surgery") else None,
    # 儿童患者特殊标记
    lambda name, data: f"👶 {name}: 儿童患者（{data['props']['age']}岁），需使用儿童剂量"
        if data["class"] == "Patient" and data["props"].get("age", 99) < 12 else None,
    # 有过敏史的患者
    lambda name, data: f"⚠️ {name}: 有 {len(data['props'].get('allergies', []))} 种药物过敏，开处方需格外注意"
        if data["class"] == "Patient" and len(data["props"].get("allergies", [])) > 0 else None,
]

hospital.reason(rules)

# 模拟处方安全检查
print("\n" + "="*50)
print("💊 模拟处方安全检查")
print("="*50)

def check_prescription(patient_name, medications, instances):
    patient = instances.get(patient_name)
    if not patient:
        print(f"  ❌ 未找到患者 {patient_name}")
        return
    allergies = set(patient["props"].get("allergies", []))
    meds = set(medications)
    conflicts = allergies & meds
    
    print(f"\n  📋 为 {patient_name} 开具处方：{medications}")
    print(f"     已知过敏：{list(allergies) if allergies else '无'}")
    if conflicts:
        print(f"     ❌ 处方被拦截！药物冲突：{list(conflicts)}")
        print(f"     💡 建议：请更换为非 {list(conflicts)} 类药物")
    else:
        print(f"     ✅ 处方安全检查通过")

check_prescription("张三", ["青霉素", "布洛芬"], hospital._instances)
check_prescription("张三", ["阿莫西林", "布洛芬"], hospital._instances)
check_prescription("李四", ["头孢", "止痛药"], hospital._instances)
check_prescription("赵六", ["头孢", "退烧药"], hospital._instances)

---
## 📦 案例 2：电商供应链

> **场景：** 一家跨境电商需要管理全球仓库、供应商、商品、订单的供应链。
>
> **核心问题：** 如何自动检测库存不足？如何根据供应商和仓库距离优化配送？

In [ ]:
# 📦 案例 2：电商供应链 OPERA 建模

supply = OntologyBuilder("电商供应链")

# O
supply.add_class("Product", desc="商品")
supply.add_class("Category", desc="商品类目")
supply.add_class("Supplier", desc="供应商")
supply.add_class("Warehouse", desc="仓库")
supply.add_class("Order", desc="订单")
supply.add_class("Shipment", desc="物流单")
supply.add_class("Customer", desc="客户")
supply.add_class("Region", desc="地区")

# P
supply.add_property("Product", "sku", "string", True, "SKU编号")
supply.add_property("Product", "name", "string", True, "商品名")
supply.add_property("Product", "price", "float", True, "单价")
supply.add_property("Product", "stock", "int", True, "库存数量")
supply.add_property("Product", "min_stock", "int", True, "最低安全库存")
supply.add_property("Warehouse", "location", "string", True, "所在城市")
supply.add_property("Warehouse", "capacity", "int", True, "最大容量")
supply.add_property("Order", "total", "float", True, "订单总额")
supply.add_property("Order", "status", "enum", True, "订单状态")

# E
supply.add_action("PlaceOrder", "Customer", "Order", "商品在售+库存>0", "创建订单+扣减库存")
supply.add_action("Restock", "System", "Product+Supplier", "库存<安全线", "生成采购单给供应商")
supply.add_event("LowStock", "stock < min_stock", "自动触发 Restock + 通知运营")
supply.add_event("DeliveryDelay", "物流超过预计时间2天", "通知客服+升级处理")

# R
supply.add_relation("belongs_to_cat", "Product", "Category", "多对一", "商品所属类目")
supply.add_relation("supplied_by", "Product", "Supplier", "多对多", "供应商供货")
supply.add_relation("stored_in", "Product", "Warehouse", "多对多", "商品存放仓库")
supply.add_relation("placed_by", "Order", "Customer", "多对一", "下单客户")
supply.add_relation("contains", "Order", "Product", "多对多", "包含商品")
supply.add_relation("fulfilled_from", "Order", "Warehouse", "多对一", "从哪个仓库发货")
supply.add_relation("shipped_via", "Order", "Shipment", "一对一", "物流信息")
supply.add_relation("located_in", "Warehouse", "Region", "多对一", "仓库所在地区")

# A
supply.add_axiom("约束", "库存非负", "Product.stock ≥ 0")
supply.add_axiom("约束", "订单金额正数", "Order.total > 0")
supply.add_axiom("推理", "自动补货", "stock < min_stock → 触发 LowStock 事件")
supply.add_axiom("推理", "就近发货", "选择距离客户最近的有库存仓库发货")

supply.print_opera()

In [ ]:
# 📦 电商供应链 — 库存管理推理实战

supply.add_instance("iPhone16", "Product", stock=5, min_stock=20, name="iPhone 16", price=7999)
supply.add_instance("MacBookPro", "Product", stock=50, min_stock=10, name="MacBook Pro", price=14999)
supply.add_instance("AirPods", "Product", stock=3, min_stock=30, name="AirPods Pro", price=1899)
supply.add_instance("iPad", "Product", stock=100, min_stock=15, name="iPad Air", price=4799)

# 库存安全检查
rules = [
    lambda name, data: (
        f"🔴 {data['props']['name']}: 库存告急！当前 {data['props']['stock']} / 安全线 {data['props']['min_stock']}"
        f" → 需补货 {data['props']['min_stock'] * 2 - data['props']['stock']} 件"
    ) if data["class"] == "Product" and data["props"].get("stock", 999) < data["props"].get("min_stock", 0) else None,
    
    lambda name, data: (
        f"🟢 {data['props']['name']}: 库存充足 ({data['props']['stock']} / {data['props']['min_stock']})"
    ) if data["class"] == "Product" and data["props"].get("stock", 0) >= data["props"].get("min_stock", 0) else None,
]

supply.reason(rules)

# 模拟下单过程
print("\n" + "="*50)
print("🛒 模拟下单流程")
print("="*50)

def simulate_order(product_name, quantity, instances):
    product = instances.get(product_name)
    if not product:
        print(f"  ❌ 商品不存在：{product_name}")
        return
    
    name = product["props"]["name"]
    stock = product["props"]["stock"]
    min_stock = product["props"]["min_stock"]
    price = product["props"]["price"]
    
    print(f"\n  📦 {name} × {quantity} (单价 ¥{price})")
    print(f"     当前库存: {stock}")
    
    if stock >= quantity:
        new_stock = stock - quantity
        product["props"]["stock"] = new_stock
        print(f"     ✅ 下单成功！总计 ¥{price * quantity}")
        print(f"     库存更新: {stock} → {new_stock}")
        
        if new_stock < min_stock:
            print(f"     🔔 触发 LowStock 事件！库存 {new_stock} < 安全线 {min_stock}")
            print(f"     📋 自动生成采购单：补货 {min_stock * 2 - new_stock} 件")
    else:
        print(f"     ❌ 库存不足！需要 {quantity} 件，但只有 {stock} 件")

simulate_order("iPhone16", 3, supply._instances)
simulate_order("MacBookPro", 2, supply._instances)
simulate_order("AirPods", 10, supply._instances)

---
## 🚦 案例 3：智慧交通

> **场景：** 城市交通管理中心需要统一管理路段、信号灯、车辆、交通事件。
>
> **核心问题：** 如何自动检测拥堵？如何为紧急车辆自动开辟绿色通道？

In [ ]:
# 🚦 案例 3：智慧交通 OPERA 建模

traffic = OntologyBuilder("智慧交通")

# O
traffic.add_class("RoadSegment", desc="路段")
traffic.add_class("Highway", parent="RoadSegment", desc="高速公路")
traffic.add_class("CityRoad", parent="RoadSegment", desc="城市道路")
traffic.add_class("TrafficLight", desc="信号灯")
traffic.add_class("Vehicle", desc="车辆")
traffic.add_class("EmergencyVehicle", parent="Vehicle", desc="紧急车辆")
traffic.add_class("Intersection", desc="交叉路口")
traffic.add_class("TrafficIncident", desc="交通事件")
traffic.add_class("Sensor", desc="传感器")

# P
traffic.add_property("RoadSegment", "speed_limit", "int", True, "限速(km/h)")
traffic.add_property("RoadSegment", "current_speed", "float", True, "当前均速")
traffic.add_property("RoadSegment", "vehicle_count", "int", True, "当前车辆数")
traffic.add_property("RoadSegment", "capacity", "int", True, "设计通行容量")
traffic.add_property("TrafficLight", "state", "enum[red,yellow,green]", True, "当前状态")
traffic.add_property("TrafficLight", "green_duration", "int", True, "绿灯持续秒数")
traffic.add_property("Vehicle", "type", "string", True, "车辆类型")
traffic.add_property("Vehicle", "speed", "float", True, "当前速度")

# E
traffic.add_action("AdjustSignal", "System", "TrafficLight", "拥堵等级>阈值", "延长绿灯时间")
traffic.add_action("EmergencyOverride", "EmergencyVehicle", "TrafficLight", "紧急车辆接近", "强制切换为绿灯")
traffic.add_event("CongestionDetected", "vehicle_count/capacity > 0.8", "触发信号调整+通知路况平台")
traffic.add_event("AccidentReport", "传感器检测到碰撞", "通知交警+调整周边信号+发布避让建议")

# R
traffic.add_relation("connects", "RoadSegment", "Intersection", "多对多", "路段连接路口")
traffic.add_relation("controlled_by", "Intersection", "TrafficLight", "一对多", "路口的信号灯")
traffic.add_relation("on_road", "Vehicle", "RoadSegment", "多对一", "车在哪条路上")
traffic.add_relation("monitored_by", "RoadSegment", "Sensor", "一对多", "路段传感器")
traffic.add_relation("incident_on", "TrafficIncident", "RoadSegment", "多对一", "事件发生路段")

# A
traffic.add_axiom("推理", "拥堵检测", "vehicle_count/capacity > 0.8 → state=congested")
traffic.add_axiom("推理", "紧急车辆优先", "EmergencyVehicle approaching → 前方所有路口绿灯")
traffic.add_axiom("约束", "速度限制", "Vehicle.speed ≤ RoadSegment.speed_limit")
traffic.add_axiom("推理", "事故周边分流", "incident_on(road) → 相邻路段信号调整分流")

traffic.print_opera()

In [ ]:
# 🚦 智慧交通 — 拥堵检测与信号调整推理

traffic.add_instance("长安街东段", "CityRoad", vehicle_count=450, capacity=500, current_speed=15, speed_limit=60)
traffic.add_instance("三环北路", "Highway", vehicle_count=200, capacity=800, current_speed=85, speed_limit=100)
traffic.add_instance("中关村大街", "CityRoad", vehicle_count=380, capacity=400, current_speed=8, speed_limit=50)
traffic.add_instance("二环南路", "CityRoad", vehicle_count=100, capacity=600, current_speed=55, speed_limit=60)
traffic.add_instance("救护车-A01", "EmergencyVehicle", type="救护车", speed=80, approaching="长安街东段")

rules = [
    # 拥堵检测
    lambda name, data: (
        f"🔴 {name}: 严重拥堵！负荷 {data['props']['vehicle_count']}/{data['props']['capacity']} "
        f"= {data['props']['vehicle_count']/data['props']['capacity']*100:.0f}%，均速 {data['props']['current_speed']}km/h"
    ) if data["class"] in ("CityRoad", "Highway") 
      and data["props"].get("vehicle_count", 0) / max(data["props"].get("capacity", 1), 1) > 0.8 else None,
    
    # 畅通检测
    lambda name, data: (
        f"🟢 {name}: 畅通 ({data['props']['vehicle_count']}/{data['props']['capacity']} "
        f"= {data['props']['vehicle_count']/data['props']['capacity']*100:.0f}%)"
    ) if data["class"] in ("CityRoad", "Highway") 
      and data["props"].get("vehicle_count", 0) / max(data["props"].get("capacity", 1), 1) <= 0.5 else None,
    
    # 紧急车辆优先
    lambda name, data: (
        f"🚑 {name}: 紧急车辆接近 {data['props'].get('approaching')}！前方信号灯强制切换为绿灯"
    ) if data["class"] == "EmergencyVehicle" and data["props"].get("approaching") else None,
]

traffic.reason(rules)

---
## 📚 案例 4：企业知识管理

> **场景：** 一家科技公司需要管理员工技能、项目经验、文档知识之间的关联。
>
> **核心问题：** 如何自动找到某个技术领域的专家？如何根据项目需求推荐合适的人选？

In [ ]:
# 📚 案例 4：企业知识管理 OPERA 建模

knowledge = OntologyBuilder("企业知识管理")

# O
knowledge.add_class("Employee", desc="员工")
knowledge.add_class("Skill", desc="技能")
knowledge.add_class("Project", desc="项目")
knowledge.add_class("Document", desc="文档")
knowledge.add_class("Team", desc="团队")
knowledge.add_class("Technology", desc="技术栈")

# P
knowledge.add_property("Employee", "name", "string", True, "姓名")
knowledge.add_property("Employee", "title", "string", True, "职位")
knowledge.add_property("Employee", "years_exp", "int", True, "工作年限")
knowledge.add_property("Skill", "level", "enum[初级,中级,高级,专家]", True, "熟练程度")
knowledge.add_property("Project", "status", "enum[规划,进行中,完成]", True, "状态")
knowledge.add_property("Document", "doc_type", "enum[设计文档,API文档,教程,复盘]", True, "文档类型")

# E
knowledge.add_action("FindExpert", "System", "Employee列表", "输入技能/技术名称", "返回匹配的专家排序列表")
knowledge.add_action("RecommendTeam", "System", "Employee列表", "输入项目技术需求", "返回推荐的团队成员组合")
knowledge.add_event("SkillGap", "项目需要的技能无人掌握", "标记技能缺口+推荐培训")

# R
knowledge.add_relation("has_skill", "Employee", "Skill", "多对多", "员工掌握技能")
knowledge.add_relation("works_on", "Employee", "Project", "多对多", "参与项目")
knowledge.add_relation("authored", "Employee", "Document", "多对多", "编写文档")
knowledge.add_relation("uses_tech", "Project", "Technology", "多对多", "项目使用技术")
knowledge.add_relation("related_to", "Skill", "Technology", "多对多", "技能关联技术")
knowledge.add_relation("belongs_to", "Employee", "Team", "多对一", "所属团队")
knowledge.add_relation("about", "Document", "Technology", "多对多", "文档涉及技术")

# A
knowledge.add_axiom("推理", "专家识别", "has_skill(emp, skill) ∧ skill.level='专家' ∧ years_exp>5 → Expert(emp, skill)")
knowledge.add_axiom("推理", "隐含技能推理", "works_on(emp, proj) ∧ uses_tech(proj, tech) → impliedSkill(emp, tech)")
knowledge.add_axiom("推理", "文档推荐", "需要了解tech → 推荐 about(doc, tech) 的所有文档")

knowledge.print_opera()

In [ ]:
# 📚 企业知识管理 — 专家发现与团队推荐实战

# 添加员工数据
employees = {
    "张工": {"title": "高级工程师", "years_exp": 8, "skills": {"Python": "专家", "Ontology": "高级", "NLP": "高级"}},
    "李工": {"title": "架构师", "years_exp": 12, "skills": {"Java": "专家", "微服务": "专家", "Kubernetes": "高级"}},
    "王工": {"title": "数据科学家", "years_exp": 5, "skills": {"Python": "高级", "ML": "专家", "NLP": "中级"}},
    "赵工": {"title": "前端工程师", "years_exp": 3, "skills": {"React": "高级", "TypeScript": "高级", "CSS": "中级"}},
    "陈工": {"title": "运维工程师", "years_exp": 6, "skills": {"Kubernetes": "专家", "Docker": "专家", "Linux": "高级"}},
}

for name, data in employees.items():
    knowledge.add_instance(name, "Employee", **data)

# 🔍 Action: 查找专家
def find_expert(skill_name, employees_data):
    """查找某个技能的专家"""
    print(f"\n🔍 查找技能专家：{skill_name}")
    print("-" * 45)
    
    results = []
    for name, data in employees_data.items():
        if data["class"] != "Employee":
            continue
        skills = data["props"].get("skills", {})
        if skill_name in skills:
            level = skills[skill_name]
            score = {"专家": 4, "高级": 3, "中级": 2, "初级": 1}.get(level, 0)
            score += data["props"].get("years_exp", 0) * 0.1  # 经验加分
            results.append((name, data["props"]["title"], level, score))
    
    results.sort(key=lambda x: -x[3])
    if results:
        for name, title, level, score in results:
            star = "⭐" * int(score)
            print(f"  {star} {name} ({title}) — {skill_name} {level}")
    else:
        print(f"  ⚠️ 未找到掌握 {skill_name} 的员工 — 技能缺口！")
    return results

# 🏗️ Action: 团队推荐
def recommend_team(required_skills, employees_data):
    """根据项目需求推荐团队组合"""
    print(f"\n🏗️ 项目团队推荐")
    print(f"   需求技能：{required_skills}")
    print("-" * 50)
    
    team = []
    covered = set()
    
    for skill in required_skills:
        best = None
        best_score = 0
        for name, data in employees_data.items():
            if data["class"] != "Employee" or name in [t[0] for t in team]:
                continue
            skills = data["props"].get("skills", {})
            if skill in skills:
                score = {"专家": 4, "高级": 3, "中级": 2, "初级": 1}.get(skills[skill], 0)
                if score > best_score:
                    best = (name, data["props"]["title"], skill, skills[skill])
                    best_score = score
        if best:
            team.append(best)
            covered.add(skill)
    
    print("  推荐团队成员：")
    for name, title, skill, level in team:
        print(f"    ✅ {name} ({title}) — 负责 {skill} ({level})")
    
    gaps = set(required_skills) - covered
    if gaps:
        print(f"\n  ⚠️ 技能缺口：{gaps} — 建议招聘或培训！")
    
    return team

# 执行查询
find_expert("Python", knowledge._instances)
find_expert("Kubernetes", knowledge._instances)
find_expert("Rust", knowledge._instances)  # 无人掌握

print("\n" + "=" * 55)
recommend_team(["Python", "NLP", "Kubernetes", "React"], knowledge._instances)
recommend_team(["Python", "Rust", "ML"], knowledge._instances)

---
## 📊 四大案例对比总结

In [1]:
# 📊 四大案例 OPERA 对比

comparison = {
    "维度": ["O 对象类型", "P 关键属性", "E 核心动作", "R 关键关系", "A 核心公理", "推理亮点"],
    "智慧医院": [
        "Patient, Doctor, Medication, Prescription",
        "allergies, blood_type, dosage",
        "Prescribe（附安全检查）",
        "contains_med, prescribed_for",
        "过敏安全约束",
        "处方自动拦截过敏药物"
    ],
    "电商供应链": [
        "Product, Order, Warehouse, Supplier",
        "stock, min_stock, capacity",
        "PlaceOrder, Restock",
        "stored_in, fulfilled_from",
        "库存 < 安全线 → 自动补货",
        "库存动态监控+自动采购"
    ],
    "智慧交通": [
        "RoadSegment, TrafficLight, Vehicle",
        "vehicle_count, capacity, speed",
        "AdjustSignal, EmergencyOverride",
        "on_road, controlled_by",
        "负荷 > 80% → 拥堵",
        "紧急车辆自动绿波通道"
    ],
    "企业知识管理": [
        "Employee, Skill, Project, Document",
        "skills, level, years_exp",
        "FindExpert, RecommendTeam",
        "has_skill, uses_tech, authored",
        "参与项目 → 隐含技能推理",
        "自动发现技能缺口"
    ],
}

# 打印对比表
print("📊 四大案例 OPERA 对比总结")
print("=" * 90)

header = f"{'维度':<12}"
for case in ["智慧医院", "电商供应链", "智慧交通", "企业知识管理"]:
    header += f" {'│':>1} {case:<18}"
print(header)
print("-" * 90)

for i, dim in enumerate(comparison["维度"]):
    row = f"{dim:<12}"
    for case in ["智慧医院", "电商供应链", "智慧交通", "企业知识管理"]:
        val = comparison[case][i][:17]
        row += f" │ {val:<17}"
    print(row)

print("=" * 90)
print("\n💡 核心结论：")
print("   1. OPERA 框架在四个完全不同的领域都适用——确实是'万能框架'")
print("   2. O/P/R 比较直观，E 和 A 是区分'好本体'和'普通数据模型'的关键")
print("   3. 公理 (A) 是 Ontology 最有价值的部分——它让系统具备自动推理能力")

📊 四大案例 OPERA 对比总结
维度           │ 智慧医院               │ 电商供应链              │ 智慧交通               │ 企业知识管理            
------------------------------------------------------------------------------------------
O 对象类型       │ Patient, Doctor,  │ Product, Order, W │ RoadSegment, Traf │ Employee, Skill, 
P 关键属性       │ allergies, blood_ │ stock, min_stock, │ vehicle_count, ca │ skills, level, ye
E 核心动作       │ Prescribe（附安全检查）  │ PlaceOrder, Resto │ AdjustSignal, Eme │ FindExpert, Recom
R 关键关系       │ contains_med, pre │ stored_in, fulfil │ on_road, controll │ has_skill, uses_t
A 核心公理       │ 过敏安全约束            │ 库存 < 安全线 → 自动补货   │ 负荷 > 80% → 拥堵     │ 参与项目 → 隐含技能推理    
推理亮点         │ 处方自动拦截过敏药物        │ 库存动态监控+自动采购       │ 紧急车辆自动绿波通道        │ 自动发现技能缺口         

💡 核心结论：
   1. OPERA 框架在四个完全不同的领域都适用——确实是'万能框架'
   2. O/P/R 比较直观，E 和 A 是区分'好本体'和'普通数据模型'的关键
   3. 公理 (A) 是 Ontology 最有价值的部分——它让系统具备自动推理能力


---
## 📝 第四课小结

### ✅ 你已经学会了：

- [x] 用 OPERA 框架对 **智慧医院** 建模 — 用药安全推理
- [x] 用 OPERA 框架对 **电商供应链** 建模 — 库存自动管理
- [x] 用 OPERA 框架对 **智慧交通** 建模 — 拥堵检测与紧急车辆优先
- [x] 用 OPERA 框架对 **企业知识管理** 建模 — 专家发现与团队推荐
- [x] 理解了 OPERA 在不同领域的 **共性和差异**

### 🔜 下一课预告

> **第五课：Ontology × LLM** — 当本体论遇上大语言模型，探索 RAG + Ontology、AI Agent + Ontology、神经符号系统等前沿融合。

In [ ]:
# 📝 第四课小测验

questions = [
    {
        "q": "在智慧医院案例中，过敏安全检查是通过什么机制实现的？",
        "opts": {"A": "医生手动查看", "B": "数据库触发器", "C": "Ontology 公理 + 关系链推理", "D": "AI 图像识别"},
        "ans": "C",
        "why": "通过 Ontology 的关系链（患者→过敏药物、处方→包含药物）和公理（过敏药不能出现在处方中）自动检测。"
    },
    {
        "q": "OPERA 框架中，哪个构件在四个案例中差异最大、最需要定制？",
        "opts": {"A": "O（对象类型）", "B": "P（属性）", "C": "R（关系）", "D": "A（公理）和 E（事件/动作）"},
        "ans": "D",
        "why": "O/P/R 相对直观，而每个领域的业务规则(A)和操作逻辑(E)差异最大，这也是 Ontology 的核心价值所在。"
    },
    {
        "q": "企业知识管理中，'张工参与了 NLP 项目 → 张工可能懂 NLP' 属于什么推理？",
        "opts": {"A": "约束检查", "B": "关系链隐含技能推理", "C": "数据统计", "D": "模式匹配"},
        "ans": "B",
        "why": "通过 works_on(emp, proj) ∧ uses_tech(proj, tech) → impliedSkill(emp, tech) 这条推理链自动发现隐含技能。"
    }
]

score = 0
print("📝 第四课小测验（共 3 题）\n")
for i, q in enumerate(questions, 1):
    print(f"第 {i} 题：{q['q']}")
    for k, v in q["opts"].items():
        print(f"  {k}. {v}")
    ans = input("你的答案 (A/B/C/D)：").strip().upper()
    if ans == q["ans"]:
        print(f"✅ 正确！{q['why']}\n")
        score += 1
    else:
        print(f"❌ 正确答案是 {q['ans']}。{q['why']}\n")

print(f"🏆 得分：{score}/3")
if score == 3:
    print("🎉 完美！你已经能用 OPERA 对任何领域进行建模了！")
print("\n➡️ 最后一课，我们将探索 Ontology 与 LLM 的融合！")